# Heston–Merton calibration & Monte Carlo — period 2018–2019

**Period file:** **2018-01-01 → 2019-12-31**.

| Role | Ticker |
|------|--------|
| Primary | **SPY** |
| Secondary | AAPL |
| Secondary | MSFT |

**Section roles**
- **§4 Calibration only:** choose lookback / rolling, **Reestimate**, inspect estimated parameters (no Monte Carlo plots here).
- **§5 Monte Carlo only:** Start / Restart; simulated paths and history comparison.
- **§6 Optimal stopping:** after §4 (and §5 paths), LSM exercise decision on SPY American calls (risk-neutral paths from the same simulator); results in `stopping_results`.

**Estimation:** Method A only (realized-variance moments + jump threshold). No MC in calibration.

True rolling rule: at each update, re-estimate \(\hat\mu,\hat\kappa,\hat\theta,\hat\xi,\hat\rho,\hat v_0,\hat\lambda,\hat\mu_J,\hat\kappa_J\) (and \(\hat\sigma_J\)) from the current window and use them for the next MC segment. See `ROLLING_CALIBRATION.md`.


## 0. Setup


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, clear_output
import ipywidgets as widgets

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA = Path("..") / ".." / "research" / "data"
PERIOD_START = pd.Timestamp("2018-01-01")
PERIOD_END = pd.Timestamp("2019-12-31")
TICKERS = ["AAPL", "MSFT", "SPY"]
N_DAYS = 252
N_STEPS = 500  # Monte Carlo time steps per path
JUMP_THRESH = 3.0  # flag |r| > c * daily σ as a jump
MIN_WINDOW = 60  # trading days required for Method A moments
COLORS = {"AAPL": "#1f77b4", "MSFT": "#ff7f0e", "SPY": "#2ca02c"}

WINDOW_OPTIONS = {
    "3 months": pd.DateOffset(months=3),
    "6 months": pd.DateOffset(months=6),
    "1 year": pd.DateOffset(years=1),
    "2 years": pd.DateOffset(years=2),
    "5 years": pd.DateOffset(years=5),
}
ROLLING_OPTIONS = ["daily", "monthly", "none"]

prices = pd.read_csv(DATA / "equity" / "prices_clean.csv", parse_dates=["Date"]).set_index("Date").sort_index()
period_prices = prices.loc[PERIOD_START:PERIOD_END, TICKERS].copy()
log_returns_all = np.log(prices[TICKERS]).diff()

rolling = {}
cal_meta = {}

print(f"Price sample: {prices.index.min().date()} → {prices.index.max().date()}")
print(
    f"Period rows: {len(period_prices)} trading days "
    f"({period_prices.index.min().date()} → {period_prices.index.max().date()})"
)
period_prices.head()

# --- clean plotting / widget memory (important after reopen) ---
plt.close("all")
plt.ioff()


## 1. Stock price trends (2018–2019)

Adjusted close for AAPL, MSFT, and SPY (primary).


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
for ax, ticker in zip(axes, TICKERS):
    s = period_prices[ticker].dropna()
    ax.plot(s.index, s.values, color=COLORS[ticker], lw=1.4)
    role = "primary" if ticker == "SPY" else "secondary"
    ax.set_ylabel("Adj close")
    ax.set_title(f"{ticker} ({role}) — adjusted close, 2018–2019")
axes[-1].set_xlabel("Date")
fig.suptitle("Stock price trends — period 2018–2019", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
display(period_prices.describe().T[["count", "mean", "min", "max"]].round(4))


## 2. Strike prices in this period

Unique strikes \(K\) from `*_options_panel.csv` with `trading_date` in **2018-01-01 → 2019-12-31**.

> AAPL strikes are on the option/contract scale; equity adj closes are split-adjusted.


In [ ]:
for ticker in TICKERS:
    path = DATA / "options" / "processed" / f"{ticker}_options_panel.csv"
    opt = pd.read_csv(path, usecols=["trading_date", "K"], parse_dates=["trading_date"])
    m = (opt["trading_date"] >= PERIOD_START) & (opt["trading_date"] <= PERIOD_END)
    sub = opt.loc[m]
    uniq = np.sort(sub["K"].dropna().unique())
    dmin, dmax = sub["trading_date"].min(), sub["trading_date"].max()
    display(Markdown(
        f"### {ticker} — {len(uniq)} unique strikes "
        f"(options quotes {dmin.date() if pd.notna(dmin) else 'n/a'} → "
        f"{dmax.date() if pd.notna(dmax) else 'n/a'})"
    ))
    print("Strikes K:", ", ".join(f"{x:g}" for x in uniq))
    display(pd.DataFrame({"K": uniq}).T)


## 3. Estimation formulas (Heston–Merton — Method A)

**Variance**

$$dv_t = \kappa(\theta - v_t)\, dt + \xi\sqrt{v_t}\, dW_t^v$$

**Price**

$$\frac{dS_t}{S_{t-}} = (\mu - \lambda\kappa_J)\, dt + \sqrt{v_t}\, dW_t^S + (e^J - 1)\, dN_t$$

$$\mathrm{Corr}(dW_t^S, dW_t^v)=\rho$$

**Discrete MC step**

$$v_{t+\Delta t}=\max\big(0,\ v_t+\kappa(\theta-v_t)\Delta t+\xi\sqrt{v_t}\sqrt{\Delta t}\,Z_v\big)$$

$$S_{t+\Delta t}=S_t\exp\Big(\big(\mu-\lambda\kappa_J-\tfrac12 v_t\big)\Delta t+\sqrt{v_t}\sqrt{\Delta t}\,Z_S+\sum_{i=1}^{N_{\Delta t}}J_i\Big)$$

with \(\mathrm{Corr}(Z_S,Z_v)=\rho\), \(N_{\Delta t}\sim\mathrm{Poisson}(\lambda\Delta t)\), \(J_i\sim N(\mu_J,\sigma_J^2)\), and \(\kappa_J=e^{\mu_J+\sigma_J^2/2}-1\).

| Parameter | Estimator (Method A — historical moments; no RNG) |
|-----------|-----------------------------------------------------|
| \(\hat\mu\) | \(\bar r_{\text{non-jump}} \times 252\) |
| \(RV_t\) | \(r_t^2\) (daily variance proxy) |
| \(\hat\theta\) | \(\overline{RV}\times 252\) |
| \(\hat v_0\) | recent \(RV\) (≤21 days) \(\times 252\) |
| \(\hat\kappa\) | \(-\ln(\rho_1)/\Delta t\) from lag-1 autocorr of \(RV\) |
| \(\hat\xi\) | moment scale of \(\Delta v\) residuals after mean-reversion drift |
| \(\hat\rho\) | \(\mathrm{Corr}(r_t,\Delta v_t)\) |
| Jump days | \(|r_t| > c\cdot\hat\sigma_{\text{day}}\) with \(c=3\) |
| \(\hat\lambda\) | \(n_{\text{jumps}} / Y\) |
| \(\hat\mu_J,\hat\sigma_J\) | mean / std of jump-day returns |
| \(\kappa_J\) | \(e^{\hat\mu_J + \hat\sigma_J^2/2}-1\) (derived) |

**Six calibrated quantities shown in §4:** \(\mu,\theta,\kappa,\xi,\rho,\lambda\). \(v_0,\mu_J,\sigma_J,\kappa_J\) are also rolled for simulation.

**True rolling:** at each update date, re-estimate from the lookback window ending there; those params drive the next Monte Carlo segment.


## 4. Calibration only — 2018–2019

Sliders + **Reestimate**. Shows **only** the rolling parameter graphs (no tables).  
Monte Carlo vs history is in **§5** only — one pair per company.


In [ ]:
def estimate_heston_merton_params(log_rets: pd.Series, jump_thresh: float = JUMP_THRESH):
    """Method A: μ, κ, θ, ξ, ρ, v0, λ, μ_J, σ_J, κ_J from a lookback window (no RNG).

    V2 fixes:
    - Continuous variance (θ, v0) from non-jump days only so jump second
      moments are not also loaded into the Heston variance level.
    - κ, ξ from AR(1)/OLS on continuous daily RV with correct lag.
      Use Euler map κ=(1-β)/Δt (no κ=2 default, no ξ≤3 cap).
      If β≤0 (no persistence): constant-vol limit ξ→ε (do not Feller-inflate ξ).
    """
    x = log_rets.dropna().astype(float)
    n = int(x.shape[0])
    nan = (np.nan,) * 10 + (n,)
    if n < MIN_WINDOW:
        return nan

    sigma_day = float(x.std(ddof=1))
    if not np.isfinite(sigma_day) or sigma_day <= 0:
        return nan

    jump_mask = np.abs(x.values) > jump_thresh * sigma_day
    jumps = x.iloc[jump_mask]
    normal = x.iloc[~jump_mask]
    base = normal if int(normal.shape[0]) >= 2 else x
    mu = float(base.mean() * N_DAYS)

    cont = base
    rv_cont = (cont ** 2).astype(float)
    theta = float(rv_cont.mean() * N_DAYS)
    recent = rv_cont.iloc[-min(21, len(rv_cont)):]
    v0 = float(recent.mean() * N_DAYS)
    if not np.isfinite(theta) or theta <= 0:
        return nan
    if not np.isfinite(v0) or v0 <= 0:
        v0 = theta

    dt = 1.0 / N_DAYS
    y = rv_cont.astype(float)
    ar = pd.concat([y.rename("y"), y.shift(1).rename("yl")], axis=1).dropna()
    beta = np.nan
    if len(ar) >= 5:
        yl = ar["yl"].to_numpy(dtype=float)
        yy = ar["y"].to_numpy(dtype=float)
        X = np.column_stack([np.ones(len(yl)), yl])
        coef, *_ = np.linalg.lstsq(X, yy, rcond=None)
        beta = float(coef[1])

    # Euler/AR(1): β ≈ 1 - κΔt  ⇒  κ = (1-β)/Δt  (no κ=2 default)
    no_persistence = (not np.isfinite(beta)) or (beta <= 0.0)
    if no_persistence:
        kappa = float(1.0 / dt)  # β=0 ⇒ κ=252
    elif beta >= 1.0:
        kappa = 1e-6
    else:
        kappa = float((1.0 - beta) / dt)
    kappa = float(max(kappa, 1e-6))

    v_ann = (y * N_DAYS).astype(float)
    pair = pd.concat(
        [(v_ann - v_ann.shift(1)).rename("dv"), v_ann.shift(1).rename("vlag")],
        axis=1,
    ).dropna()
    if no_persistence:
        xi = 1e-6
    elif len(pair) >= 5:
        vlag_vals = pair["vlag"].to_numpy(dtype=float)
        drift = kappa * (theta - vlag_vals) * dt
        resid_v = pair["dv"].to_numpy(dtype=float) - drift
        mean_v = float(np.mean(vlag_vals))
        var_resid = float(np.var(resid_v, ddof=1))
        if mean_v > 0 and np.isfinite(var_resid) and var_resid > 0:
            xi = float(np.sqrt(var_resid / (mean_v * dt)))
        else:
            xi = 1e-6
    else:
        xi = 1e-6
    if not np.isfinite(xi) or xi < 0:
        xi = 1e-6
    xi = float(max(xi, 1e-6))
    if not no_persistence:
        feller = 2.0 * kappa * theta
        if np.isfinite(feller) and feller > 0 and xi * xi > feller:
            xi = float(np.sqrt(feller) * 0.999)

    aligned = pd.concat(
        [cont.rename("r"), (cont ** 2 * N_DAYS).diff().rename("dv")], axis=1
    ).dropna()
    if len(aligned) >= 5:
        rho = float(aligned["r"].corr(aligned["dv"]))
    else:
        rho = -0.5
    if not np.isfinite(rho):
        rho = -0.5
    rho = float(np.clip(rho, -0.99, 0.99))

    years = n / float(N_DAYS)
    n_jumps = int(jump_mask.sum())
    lam = float(n_jumps / years) if years > 0 else 0.0

    if n_jumps >= 2:
        mu_j = float(jumps.mean())
        sigma_j = float(jumps.std(ddof=1))
    elif n_jumps == 1:
        mu_j = float(jumps.iloc[0])
        sigma_j = 0.0
    else:
        mu_j = 0.0
        sigma_j = 0.0
    if not np.isfinite(sigma_j) or sigma_j < 0:
        sigma_j = 0.0

    kappa_j = float(np.exp(mu_j + 0.5 * sigma_j**2) - 1.0)
    return mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, kappa_j, n

def _slice_window(rets: pd.Series, end: pd.Timestamp, offset: pd.DateOffset) -> pd.Series:
    start = end - offset
    return rets.loc[(rets.index > start) & (rets.index <= end)]


def calibrate_ticker(ticker: str, window_label: str, rolling_mode: str) -> pd.DataFrame:
    rets = log_returns_all[ticker].dropna()
    offset = WINDOW_OPTIONS[window_label]
    rows = []

    if rolling_mode == "daily":
        update_dates = rets.loc[(rets.index >= PERIOD_START) & (rets.index <= PERIOD_END)].index
    elif rolling_mode == "monthly":
        t0 = period_prices[ticker].dropna().index[0]
        month_ends = pd.date_range(PERIOD_START, PERIOD_END, freq="ME")
        update_dates = pd.DatetimeIndex([t0]).append(month_ends).unique().sort_values()
    else:
        update_dates = pd.DatetimeIndex([period_prices[ticker].dropna().index[0]])

    for t_u in update_dates:
        window = _slice_window(rets, pd.Timestamp(t_u), offset)
        mu, kappa, theta, xi, rho, v0, lam, mu_j, sigma_j, kappa_j, n = estimate_heston_merton_params(window)
        if n < MIN_WINDOW or not np.isfinite(mu):
            continue
        rows.append({
            "date": pd.Timestamp(t_u),
            "window_start": window.index.min(),
            "window_end": window.index.max(),
            "n_days": n,
            "mu": mu,
            "kappa": kappa,
            "theta": theta,
            "xi": xi,
            "rho": rho,
            "v0": v0,
            "lam": lam,
            "mu_j": mu_j,
            "sigma_j": sigma_j,
            "kappa_j": kappa_j,
        })
    return pd.DataFrame(rows)


def _mc_time_grid(hist: pd.Series, n_steps: int = N_STEPS):
    """Evenly spaced trading-day grid with exactly n_steps steps (n_steps+1 prices)."""
    hist = hist.dropna()
    n_full = len(hist)
    if n_full <= n_steps + 1:
        return hist
    idx = np.linspace(0, n_full - 1, n_steps + 1)
    idx = np.rint(idx).astype(int)
    for i in range(1, len(idx)):
        if idx[i] <= idx[i - 1]:
            idx[i] = min(idx[i - 1] + 1, n_full - 1)
    return hist.iloc[idx]


def param_schedule_for_steps(ticker: str, cal_table: pd.DataFrame):
    hist = _mc_time_grid(period_prices[ticker], N_STEPS)
    dates = hist.index
    n_steps = len(dates) - 1
    cal = cal_table.sort_values("date").reset_index(drop=True)
    cal_dates = pd.to_datetime(cal["date"]).to_numpy()

    cols = ["mu", "kappa", "theta", "xi", "rho", "v0", "lam", "mu_j", "sigma_j", "kappa_j"]
    arrs = {c: cal[c].to_numpy(dtype=float) for c in cols}
    steps = {c: np.empty(n_steps, dtype=float) for c in cols}

    for i in range(n_steps):
        idx = np.searchsorted(cal_dates, np.datetime64(dates[i]), side="right") - 1
        if idx < 0:
            idx = 0
        for c in cols:
            steps[c][i] = arrs[c][idx]

    return (
        dates,
        steps["mu"], steps["kappa"], steps["theta"], steps["xi"], steps["rho"],
        steps["v0"], steps["lam"], steps["mu_j"], steps["sigma_j"], steps["kappa_j"],
        float(hist.iloc[0]), hist,
    )


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def plot_rolling_paths(rolling_dict: dict, window_label: str, rolling_mode: str):
    """Six rolling-parameter graphs: μ, θ, κ, ξ, ρ, λ."""
    panels = [
        ("mu", "μ̂ (annual)", "Estimated drift"),
        ("theta", "θ̂ (var)", "Long-run variance"),
        ("kappa", "κ̂", "Mean-reversion speed"),
        ("xi", "ξ̂", "Vol-of-vol"),
        ("rho", "ρ̂", "Price–vol correlation"),
        ("lam", "λ̂ (jumps/year)", "Estimated jump intensity"),
    ]
    with plt.ioff():
        fig, axes = plt.subplots(6, 1, figsize=(11, 14), sharex=True)
        for ax, (col, ylab, title) in zip(axes, panels):
            for t in TICKERS:
                r = rolling_dict[t]
                x = pd.to_datetime(r["date"])
                mark = "o" if len(r) < 40 else None
                ax.plot(x, r[col], lw=1.2, label=t, color=COLORS[t], marker=mark, ms=3)
            if col in {"mu", "rho"}:
                ax.axhline(0, color="0.5", lw=0.8)
            ax.set_ylabel(ylab)
            ax.set_title(f"{title} — {rolling_mode}, lookback {window_label}")
            ax.legend(frameon=False, ncol=3)
        axes[-1].set_xlabel("Date")
        fig.tight_layout()
    _show_fig(fig)


# Reset kernel-side UI handles so reopen + Run All cannot reuse stale widgets
plt.close("all")
plt.ioff()
rolling = {}
cal_meta = {}

cal_out = widgets.Output(layout=widgets.Layout(width="100%"))
window_slider = widgets.SelectionSlider(
    options=list(WINDOW_OPTIONS.keys()),
    value="3 months",
    description="Lookback",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
rolling_slider = widgets.SelectionSlider(
    options=ROLLING_OPTIONS,
    value="daily",
    description="Rolling",
    continuous_update=False,
    style={"description_width": "70px"},
    layout=widgets.Layout(width="420px"),
)
btn_reestimate = widgets.Button(description="Reestimate", button_style="primary", icon="refresh")

cal_ui = widgets.VBox([
    widgets.HTML(
        "<b>§4 Calibration (graphs only)</b> — lookback + rolling, then <b>Reestimate</b>. "
        "Method A moments: μ̂, θ̂, κ̂, ξ̂, ρ̂, λ̂. No Monte Carlo here."
    ),
    window_slider,
    rolling_slider,
    btn_reestimate,
    cal_out,
])


def reestimate(_=None):
    global rolling, cal_meta
    window_label = window_slider.value
    rolling_mode = rolling_slider.value
    rolling = {t: calibrate_ticker(t, window_label, rolling_mode) for t in TICKERS}
    cal_meta = {"window_label": window_label, "rolling_mode": rolling_mode}

    with cal_out:
        clear_output(wait=True)
        display(Markdown(
            f"**Calibration updated (Method A):** lookback=`{window_label}`, rolling=`{rolling_mode}` "
            f"(n_updates: " + ", ".join(f"{t}={len(rolling[t])}" for t in TICKERS) + ")"
        ))
        plot_rolling_paths(rolling, window_label, rolling_mode)
        display(Markdown(
            "Go to **§5** and click **Start** for one MC pair per company. "
            "Also estimated in the same windows (for simulation): "
            r"$v_0$, $\mu_J$, $\sigma_J$, $\kappa_J$."
        ))


btn_reestimate.on_click(reestimate)
display(cal_ui)
reestimate()


## 5. Monte Carlo only — one graph pair per company (2018–2019)

| Left | Right |
|------|--------|
| Monte Carlo paths + expected path | Expected path vs historical prices |

Uses latest **Reestimate** from §4. **Start** / **Restart** redraw that single pair (never stacks another copy).


In [ ]:
def simulate_heston_merton_rolling(
    mu_step, kappa_step, theta_step, xi_step, rho_step, v0_step,
    lam_step, muj_step, sj_step, kapj_step, S0, n_paths, seed,
):
    """Heston–Merton Euler MC; params may change by step (rolling schedule)."""
    rng = np.random.default_rng(seed)
    n_steps = len(mu_step)
    dt = 1.0 / N_DAYS
    paths = np.empty((n_paths, n_steps + 1), dtype=float)
    paths[:, 0] = S0
    v = np.full(n_paths, float(v0_step[0]), dtype=float)

    for i in range(n_steps):
        mu = mu_step[i]
        kappa = kappa_step[i]
        theta = theta_step[i]
        xi = xi_step[i]
        rho = float(np.clip(rho_step[i], -0.999, 0.999))
        lam = max(float(lam_step[i]), 0.0)
        mu_j = muj_step[i]
        sigma_j = max(float(sj_step[i]), 0.0)
        kappa_j = kapj_step[i]

        z_v = rng.standard_normal(n_paths)
        z_indep = rng.standard_normal(n_paths)
        z_s = rho * z_v + np.sqrt(max(1.0 - rho**2, 0.0)) * z_indep

        v_pos = np.maximum(v, 0.0)
        v = v + kappa * (theta - v_pos) * dt + xi * np.sqrt(v_pos) * np.sqrt(dt) * z_v
        v = np.maximum(v, 0.0)
        v_pos = np.maximum(v, 0.0)

        n_jumps = rng.poisson(lam * dt, size=n_paths)
        jump_sizes = np.zeros(n_paths, dtype=float)
        mask = n_jumps > 0
        if mask.any():
            jump_sizes[mask] = (
                n_jumps[mask] * mu_j
                + np.sqrt(n_jumps[mask]) * sigma_j * rng.standard_normal(int(mask.sum()))
            )

        paths[:, i + 1] = paths[:, i] * np.exp(
            (mu - 0.5 * v_pos - lam * kappa_j) * dt
            + np.sqrt(v_pos * dt) * z_s
            + jump_sizes
        )
    return paths


def _show_fig(fig):
    """Show a figure exactly once as PNG (avoids inline double-paint)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _draw_ticker_pair(ticker: str, out: widgets.Output, seed: int, n_paths: int = 1000):
    """Replace contents of `out` with exactly one 1×2 figure."""
    with out:
        clear_output(wait=True)
        if ticker not in rolling or len(rolling[ticker]) == 0:
            display(Markdown("Run **Reestimate** in §4 first."))
            return
        (
            dates_now, mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            lam_now, muj_now, sj_now, kapj_now, S0_now, hist_now,
        ) = param_schedule_for_steps(ticker, rolling[ticker])
        paths = simulate_heston_merton_rolling(
            mu_now, kappa_now, theta_now, xi_now, rho_now, v0_now,
            lam_now, muj_now, sj_now, kapj_now, S0_now, n_paths, seed,
        )
        expected = paths.mean(axis=0)

        with plt.ioff():
            fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2))
            axes[0].plot(dates_now, paths.T, color=COLORS[ticker], alpha=0.12, lw=0.7)
            axes[0].plot(dates_now, expected, color="black", lw=2.2, label="expected path (MC mean)")
            axes[0].set_title(f"{ticker}: Heston–Merton Monte Carlo")
            axes[0].set_ylabel("price")
            axes[0].legend(loc="best", frameon=False)

            axes[1].plot(dates_now, hist_now.values, color=COLORS[ticker], lw=1.8, label="historical")
            axes[1].plot(dates_now, expected, color="black", lw=2.0, ls="--", label="expected path")
            axes[1].set_title(f"{ticker}: expected vs history")
            axes[1].set_ylabel("price")
            axes[1].legend(loc="best", frameon=False)
            for ax in axes:
                ax.set_xlabel("date")
            fig.suptitle(
                f"{ticker} | seed={seed} | {cal_meta.get('rolling_mode')} / {cal_meta.get('window_label')} | Method A",
                fontsize=11,
                y=1.02,
            )
            fig.tight_layout()
        _show_fig(fig)
        rmse = float(np.sqrt(np.mean((expected - hist_now.values) ** 2)))
        print(f"RMSE(expected vs historical) = {rmse:.4f} | seed = {seed}")


def make_ticker_panel(ticker: str, n_paths: int = 1000):
    """One Output per company. Start/Restart only replace that Output (no stacking)."""
    state = {"seed": 42}
    mode = cal_meta.get("rolling_mode", "?")
    win = cal_meta.get("window_label", "?")
    out = widgets.Output(layout=widgets.Layout(width="100%"))
    btn_start = widgets.Button(description="Start", button_style="success", icon="play")
    btn_restart = widgets.Button(description="Restart", button_style="warning", icon="refresh")
    info = widgets.HTML(f"<b>{ticker}</b> — one graph pair | lookback={win}, mode={mode}")

    busy = {"on": False}

    def on_start(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    def on_restart(_):
        if busy["on"]:
            return
        busy["on"] = True
        try:
            state["seed"] = int(np.random.default_rng().integers(0, 1_000_000_000))
            _draw_ticker_pair(ticker, out, state["seed"], n_paths)
        finally:
            busy["on"] = False

    btn_start.on_click(on_start)
    btn_restart.on_click(on_restart)
    return widgets.VBox([info, widgets.HBox([btn_start, btn_restart]), out])


plt.close("all")
plt.ioff()

mc_host = widgets.VBox([])
children = [widgets.HTML("<b>§5 Monte Carlo — click <i>Start</i> once per company (one pair only)</b>")]
for ticker in TICKERS:
    role = "primary" if ticker == "SPY" else "secondary"
    children.append(widgets.HTML(f"<h4 style='margin:8px 0 4px'>{ticker} ({role})</h4>"))
    children.append(make_ticker_panel(ticker))
mc_host.children = tuple(children)
display(mc_host)


## 6. Optimal stopping (American calls — Heston–Merton)

Continuous with §5: after Monte Carlo stock paths are available, use the **same Heston–Merton simulator** and §4 calibration on **SPY** to decide exercise vs wait for American calls.

At each day along each path:
1. Immediate payoff: $\max(S_t - K, 0)$
2. Continuation value: Longstaff–Schwartz regression on the path cloud (basis $1, S, S^2$)
3. Exercise if payoff $>$ continuation

Paths for pricing are **risk-neutral** (drift $\mu \rightarrow r$ from the option panel; vol/jumps from §4). Not the single expected path — the full Monte Carlo cloud.

**Workflow:** §4 **Reestimate** → §5 **Start** (optional viz) → §6 **Compute stopping**.



In [ ]:
import sys
_SCRIPTS = Path("..") / "scripts"
if str(_SCRIPTS.resolve()) not in sys.path:
    sys.path.insert(0, str(_SCRIPTS.resolve()))

from american_lsm import (
    lsm_american_call,
    load_spy_calls,
    params_asof,
    sample_spy_calls,
)

def _show_fig(fig):
    """Show figure once as PNG (same pattern as §5)."""
    import io
    from IPython.display import Image
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=110, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    display(Image(data=buf.getvalue()))


def _rn_paths_for_contract(row, n_paths: int, seed: int):
    """Risk-neutral paths to expiry using §5 Heston–Merton simulator (μ → r)."""
    p = params_asof(rolling["SPY"], row.trading_date)
    if p is None:
        raise RuntimeError("No SPY calibration — run Reestimate in §4 first.")
    dte = int(row.dte)
    if dte < 2:
        raise ValueError("dte must be >= 2")
    r = float(row.r)
    S0 = float(row.S_t)
    mu_step = np.full(dte, r, dtype=float)
    kappa_step = np.full(dte, float(p["kappa"]), dtype=float)
    theta_step = np.full(dte, float(p["theta"]), dtype=float)
    xi_step = np.full(dte, float(p["xi"]), dtype=float)
    rho_step = np.full(dte, float(p["rho"]), dtype=float)
    v0_step = np.full(dte, float(p["v0"]), dtype=float)
    lam_step = np.full(dte, float(p["lam"]), dtype=float)
    muj_step = np.full(dte, float(p["mu_j"]), dtype=float)
    sj_step = np.full(dte, float(p["sigma_j"]), dtype=float)
    kapj_step = np.full(dte, float(p["kappa_j"]), dtype=float)
    return simulate_heston_merton_rolling(
        mu_step, kappa_step, theta_step, xi_step, rho_step, v0_step,
        lam_step, muj_step, sj_step, kapj_step, S0, n_paths, seed,
    )


_spy_calls_all = load_spy_calls(DATA)
_contracts = sample_spy_calls(
    _spy_calls_all, PERIOD_START, PERIOD_END, n_total=24, seed=42
)
stopping_results = None

display(Markdown(
    f"Sampled **{len(_contracts)}** SPY American calls in "
    f"{PERIOD_START.date()} → {PERIOD_END.date()} "
    f"(ATM band / DTE 7–60 as in the panel)."
))
if len(_contracts):
    display(
        _contracts[
            ["trading_date", "S_t", "K", "dte", "r", "moneyness", "option_price"]
        ].head(12)
    )

_stop_n_paths = widgets.IntSlider(
    value=2000, min=500, max=8000, step=500, description="n_paths",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="360px"),
)
_stop_seed = widgets.IntText(value=42, description="seed", layout=widgets.Layout(width="200px"))
_btn_stop = widgets.Button(
    description="Compute stopping", button_style="primary", icon="calculator"
)
_stop_out = widgets.Output(layout=widgets.Layout(width="100%"))
_stop_busy = {"on": False}


def _run_optimal_stopping(_=None):
    global stopping_results
    if _stop_busy["on"]:
        return
    _stop_busy["on"] = True
    with _stop_out:
        clear_output(wait=True)
        try:
            if "SPY" not in rolling or len(rolling["SPY"]) == 0:
                display(Markdown("Run **Reestimate** in §4 first (need SPY calibration)."))
                return
            if _contracts is None or len(_contracts) == 0:
                display(Markdown("No SPY call contracts in this period panel slice."))
                return

            n_paths = int(_stop_n_paths.value)
            seed0 = int(_stop_seed.value)
            rows = []
            example = None
            dt = 1.0 / N_DAYS

            for i, row in enumerate(_contracts.itertuples(index=False)):
                paths = _rn_paths_for_contract(row, n_paths, seed0 + i)
                res = lsm_american_call(paths, K=float(row.K), r=float(row.r), dt=dt)
                err = res.price - float(row.option_price)
                rows.append({
                    "trading_date": row.trading_date,
                    "S_t": float(row.S_t),
                    "K": float(row.K),
                    "dte": int(row.dte),
                    "r": float(row.r),
                    "market": float(row.option_price),
                    "model_price": res.price,
                    "error": err,
                    "early_ex_frac": res.early_exercise_frac,
                    "mean_ex_day": res.mean_exercise_step,
                })
                if example is None:
                    example = (row, paths, res)

            stopping_results = pd.DataFrame(rows)
            rmse = float(np.sqrt(np.mean(stopping_results["error"] ** 2)))
            mae = float(np.mean(np.abs(stopping_results["error"])))

            display(Markdown(
                f"### Heston–Merton — LSM results (SPY)\n"
                f"n_paths={n_paths} | contracts={len(stopping_results)} | "
                f"RMSE={rmse:.4f} | MAE={mae:.4f} | "
                f"mean early-exercise fraction="
                f"{stopping_results['early_ex_frac'].mean():.3f}"
            ))
            display(
                stopping_results[
                    ["trading_date", "S_t", "K", "dte", "market", "model_price",
                     "error", "early_ex_frac", "mean_ex_day"]
                ].round(4)
            )

            with plt.ioff():
                fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
                ax = axes[0]
                ax.scatter(
                    stopping_results["market"], stopping_results["model_price"],
                    alpha=0.75, color=COLORS.get("SPY", "#2ca02c"),
                )
                lo = min(stopping_results["market"].min(), stopping_results["model_price"].min())
                hi = max(stopping_results["market"].max(), stopping_results["model_price"].max())
                ax.plot([lo, hi], [lo, hi], "k--", lw=1)
                ax.set_xlabel("market option_price")
                ax.set_ylabel("model LSM price")
                ax.set_title("Price: model vs market")

                axes[1].bar(
                    ["model", "market"],
                    [stopping_results["model_price"].mean(), stopping_results["market"].mean()],
                    color=[COLORS.get("SPY", "#2ca02c"), "#7f7f7f"],
                )
                axes[1].set_title("Mean option value")
                axes[1].set_ylabel("price")

                axes[2].hist(
                    stopping_results["mean_ex_day"], bins=12,
                    color=COLORS.get("SPY", "#2ca02c"), alpha=0.85, edgecolor="white",
                )
                axes[2].set_xlabel("mean exercise day (by contract)")
                axes[2].set_title("Optimal exercise timing")
                fig.suptitle(
                    f"Heston–Merton optimal stopping | {cal_meta.get('rolling_mode')} / "
                    f"{cal_meta.get('window_label')}",
                    fontsize=11, y=1.02,
                )
                fig.tight_layout()
            _show_fig(fig)

            if example is not None:
                row, paths, res = example
                j = int(np.argmin(np.abs(res.exercise_steps - res.mean_exercise_step)))
                t_ex = int(res.exercise_steps[j])
                with plt.ioff():
                    fig2, ax = plt.subplots(figsize=(10, 3.8))
                    ax.plot(paths[j], color=COLORS.get("SPY", "#2ca02c"), lw=1.5, label="one RN path")
                    ax.axhline(float(row.K), color="gray", ls="--", lw=1, label=f"K={row.K:g}")
                    ax.scatter(
                        [t_ex], [paths[j, t_ex]], color="crimson", zorder=5, s=50,
                        label=f"exercise day {t_ex}",
                    )
                    ax.set_xlabel("day")
                    ax.set_ylabel("S")
                    ax.set_title(
                        f"Example path | trade {pd.Timestamp(row.trading_date).date()} | "
                        f"dte={int(row.dte)} | model={res.price:.3f} vs mkt={float(row.option_price):.3f}"
                    )
                    ax.legend(frameon=False, loc="best")
                    fig2.tight_layout()
                _show_fig(fig2)

            display(Markdown(
                "Results stored in `stopping_results` "
                "(model_price, error, early_ex_frac, mean_ex_day)."
            ))
        except Exception as exc:
            display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))
        finally:
            _stop_busy["on"] = False


_btn_stop.on_click(_run_optimal_stopping)
display(widgets.VBox([
    widgets.HTML("<b>§6 Optimal stopping — SPY American calls (LSM)</b>"),
    widgets.HBox([_stop_n_paths, _stop_seed, _btn_stop]),
    _stop_out,
]))



## 7. Reminder

1. **§4:** sliders → **Reestimate** → read parameter tables / rolling charts.
2. **§5:** **Start** → Monte Carlo stock paths + expected vs history (one pair per ticker).
3. **§6:** **Compute stopping** → LSM exercise decision + model vs market on SPY calls (needs §4).
4. **Restart** (§5) only changes the random seed for path plots.

